# Evaluate runs and build the paper tables

Three modes. `quick`: recompute every table and figure of the selected runs from their saved per-row predictions (CPU, no data needed). `full`: rebuild the test split and run the bundles again (needs the data; GPU optional). `tables`: aggregate all completed runs across seeds (means, SDs, paired differences, significance, recommended seeds) and write the paper tables and figures.

* **Runtime:** minutes (quick, tables); full depends on the test split size
* **GPU:** not needed (optional for full)
* **Before running:** follow [docs/DATA.md](https://github.com/selimsidan/dmoe_nids/blob/main/docs/DATA.md) to put the four NF-v3 CSVs in your Drive (or set `DATA_ROOT`).
* **How to use:** edit only the SETTINGS cell, then *Runtime > Run all*. Rerunning after a disconnect continues where training stopped.


In [ ]:
# ============== SETTINGS: the only cell you edit ==============
REPO_REF    = "main"     # branch or release tag (e.g. "v1.0.0") of the repository
STORAGE     = "drive"    # "drive": Google Drive (persists) | "local": /content (lost at disconnect)
DATA_ROOT   = None       # None: Colab secret NIDS_DATA_DIR, else MyDrive/NIDS_datasets
OUTPUT_ROOT = None       # None: Colab secret NIDS_OUTPUT_DIR, else MyDrive/NIDS_outputs
# Experiment
MODE       = "tables"   # "quick" | "full" | "tables"
CONDITIONS = ["M", "B1", "B2", "B3", "A1", "A2", "A3"]
SEEDS      = [0, 1, 2, 3, 4]   # or "recommended" (quick/full: each condition's recommended seed)
BUNDLE_SOURCE = "local"   # full mode: "local" (runs on your storage) | "hub" (published bundles)
# ===============================================================

In [ ]:
# Setup: clone the repository, install it, mount storage and resolve the roots.
import os, subprocess, sys

REPO_URL, TARGET = "https://github.com/selimsidan/dmoe_nids.git", "/content/dmoe_nids"

def _git(*args, cwd=None, env=None):
    subprocess.run(["git", *args], cwd=cwd, check=True, capture_output=True, text=True,
                   env={**os.environ, **(env or {})})

def _fetch(env=None):
    if os.path.isdir(os.path.join(TARGET, ".git")):
        _git("fetch", "--quiet", "--tags", "origin", cwd=TARGET, env=env)
    else:
        _git("clone", "--quiet", REPO_URL, TARGET, env=env)

try:
    _fetch()
except subprocess.CalledProcessError:
    # Only for a private repository or fork: retry with the Colab secret GITHUB_TOKEN (never printed).
    import base64
    from google.colab import userdata
    _token = base64.b64encode(f"x-access-token:{userdata.get('GITHUB_TOKEN')}".encode()).decode()
    _fetch({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
            "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {_token}"})
    del _token
_git("checkout", "--quiet", REPO_REF, cwd=TARGET)
if subprocess.run(["git", "symbolic-ref", "-q", "HEAD"], cwd=TARGET).returncode == 0:
    # Fast-forward to the commits fetched above (no second download, so no token needed).
    _git("merge", "--quiet", "--ff-only", f"origin/{REPO_REF}", cwd=TARGET)

# Installs the package and any missing dependencies; Colab's own torch build is kept.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", TARGET], check=True)
sys.path.insert(0, os.path.join(TARGET, "src"))
os.environ["PYTHONPATH"] = os.path.join(TARGET, "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
os.chdir(TARGET)

import dmoe.colab
ROOTS = dmoe.colab.setup(STORAGE, DATA_ROOT, OUTPUT_ROOT, require_data=(MODE == 'full'))

In [ ]:
import os, pandas as pd
from IPython.display import Image, display
from dmoe.evaluation.aggregate import aggregate
from dmoe.evaluation.evaluate import evaluate
from dmoe.evaluation.paper_tables import make_paper_tables
from dmoe.training.campaign import load_campaign

OUT = ROOTS['output'].path
CAMPAIGN = load_campaign()

def selected_runs():
    if SEEDS == 'recommended':
        rec = pd.read_csv(os.path.join(OUT, 'summary', 'Recommended_Seeds.csv'))
        return [(r.condition, int(r.recommended_seed)) for r in rec.itertuples() if r.condition in CONDITIONS]
    return [(c, s) for c in CONDITIONS for s in SEEDS]

if MODE == 'tables':
    aggregate(OUT, conditions=CONDITIONS, resamples=CAMPAIGN.bootstrap_resamples,
              aggregation_seed=CAMPAIGN.aggregation_seed)
    paper = make_paper_tables(OUT)
    display(pd.read_csv(os.path.join(OUT, 'summary', 'Summary_Overall.csv')).query("metric == 'macro_f1'"))
    for table in ('Table_Main', 'Table_Ablations'):
        path = os.path.join(paper, table + '.csv')
        if os.path.isfile(path):
            print(table); display(pd.read_csv(path))
    figure = os.path.join(paper, 'Figure_Confusion_M.png')
    if os.path.isfile(figure):
        display(Image(figure, width=900))
elif MODE == 'quick':
    for condition, seed in selected_runs():
        run = os.path.join(OUT, 'runs', condition, f'seed{seed}', 'eval', 'test')
        npz = os.path.join(run, 'Test_Predictions.npz')
        if not os.path.isfile(npz):
            print(f'{condition} seed {seed}: no saved predictions, skipped'); continue
        frames = evaluate(os.path.join(OUT, 'reevaluation', condition, f'seed{seed}'), predictions=npz,
                          condition=condition, seed=seed, split='test')
        print(condition, 'seed', seed); display(frames['overall'][['origin', 'macro_f1', 'accuracy', 'weighted_f1']])
elif MODE == 'full':
    from dmoe.inference.bundle import load_bundle
    from dmoe.training.campaign import resolve_unit_config
    from dmoe.training.out_of_core_data import prepare_out_of_core_data
    context = prepare_out_of_core_data(resolve_unit_config(CAMPAIGN, 'M', 0, output_root=OUT))
    for condition, seed in selected_runs():
        source = f'hf:{condition}/seed{seed}' if BUNDLE_SOURCE == 'hub' else os.path.join(OUT, 'runs', condition, f'seed{seed}', 'bundle')
        bundle = load_bundle(source, 'cuda' if __import__('torch').cuda.is_available() else 'cpu')
        if bundle.preprocessing['signature'] != context.preprocessing_signature:
            raise SystemExit(f'{condition} seed {seed}: trained on a different split; refusing to evaluate')
        frames = evaluate(os.path.join(OUT, 'reevaluation_full', condition, f'seed{seed}'), bundle=bundle,
                          context=context, split='test', condition=condition, seed=seed)
        print(condition, 'seed', seed); display(frames['overall'][['origin', 'macro_f1', 'accuracy', 'weighted_f1']])
else:
    raise ValueError("MODE must be 'quick', 'full' or 'tables'")